In [1]:
!wget https://archive.ics.uci.edu/static/public/468/online+shoppers+purchasing+intention+dataset.zip
!unzip -o online+shoppers+purchasing+intention+dataset.zip

--2026-11-17 14:21:10--  https://archive.ics.uci.edu/static/public/468/online+shoppers+purchasing+intention+dataset.zip
Resolving archive.ics.uci.edu (archive.ics.uci.edu)... 128.195.10.252
Connecting to archive.ics.uci.edu (archive.ics.uci.edu)|128.195.10.252|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: unspecified
Saving to: ‘online+shoppers+purchasing+intention+dataset.zip’

online+shoppers+pur     [ <=>                ]   1.02M  6.14MB/s    in 0.2s    

2026-11-17 14:21:10 (6.14 MB/s) - ‘online+shoppers+purchasing+intention+dataset.zip’ saved [1072219]

Archive:  online+shoppers+purchasing+intention+dataset.zip
 extracting: online_shoppers_intention.csv  


In [2]:
!pip install -q optuna

In [3]:
import pandas as pd
import numpy as np

# Direct raw CSV mirror of the exact UCI Online Shoppers dataset
url = "https://raw.githubusercontent.com/defcom17/NSL_KDD/master/online_shoppers_intention.csv"
# Backup reliable mirror:
url_mirror = "https://raw.githubusercontent.com/sharmaroshan/Online-Shoppers-Purchasing-Intention/master/online_shoppers_intention.csv"

In [4]:
import os
import joblib
import optuna
import zipfile
import torch
import torch.nn as nn
import pandas as pd
from sklearn.preprocessing import OneHotEncoder , StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.metrics import classification_report , confusion_matrix , roc_auc_score
from torch.utils.data import Dataset , DataLoader

In [5]:
df = pd.read_csv('/content/online_shoppers_intention.csv')
print(df.shape)
df.head(5)

(12330, 18)


,Administrative,Administrative_Duration,Informational,Informational_Duration,ProductRelated,ProductRelated_Duration,BounceRates,ExitRates,PageValues,SpecialDay,Month,OperatingSystems,Browser,Region,TrafficType,VisitorType,Weekend,Revenue
0,0,0.0,0,0.0,1,0.000000,0.20,0.20,0.0,0.0,Feb,1,1,1,1,Returning_Visitor,False,False
1,0,0.0,0,0.0,2,64.000000,0.00,0.10,0.0,0.0,Feb,2,2,1,2,Returning_Visitor,False,False
2,0,0.0,0,0.0,1,0.000000,0.20,0.20,0.0,0.0,Feb,4,1,9,3,Returning_Visitor,False,False
3,0,0.0,0,0.0,2,2.666667,0.05,0.14,0.0,0.0,Feb,3,2,2,4,Returning_Visitor,False,False
4,0,0.0,0,0.0,10,627.500000,0.02,0.05,0.0,0.0,Feb,3,3,1,4,Returning_Visitor,True,False


In [6]:
df.isnull().sum()

,0
Administrative,0
Administrative_Duration,0
Informational,0
Informational_Duration,0
ProductRelated,0
ProductRelated_Duration,0
BounceRates,0
ExitRates,0
PageValues,0
SpecialDay,0


In [8]:
df[df.duplicated()]

,Administrative,Administrative_Duration,Informational,Informational_Duration,ProductRelated,ProductRelated_Duration,BounceRates,ExitRates,PageValues,SpecialDay,Month,OperatingSystems,Browser,Region,TrafficType,VisitorType,Weekend,Revenue
158,0,0.0,0,0.0,1,0.0,0.2,0.2,0.0,0.0,Feb,1,1,1,3,Returning_Visitor,False,False
159,0,0.0,0,0.0,1,0.0,0.2,0.2,0.0,0.0,Feb,3,2,3,3,Returning_Visitor,False,False
178,0,0.0,0,0.0,1,0.0,0.2,0.2,0.0,0.0,Feb,3,2,3,3,Returning_Visitor,False,False
418,0,0.0,0,0.0,1,0.0,0.2,0.2,0.0,0.0,Mar,1,1,1,1,Returning_Visitor,True,False
456,0,0.0,0,0.0,1,0.0,0.2,0.2,0.0,0.0,Mar,2,2,4,1,Returning_Visitor,False,False
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
11934,0,0.0,0,0.0,1,0.0,0.2,0.2,0.0,0.0,Dec,1,1,1,2,New_Visitor,False,False
11938,0,0.0,0,0.0,1,0.0,0.2,0.2,0.0,0.0,Dec,1,1,4,1,Returning_Visitor,True,False
12159,0,0.0,0,0.0,1,0.0,0.2,0.2,0.0,0.0,Dec,1,1,1,3,Returning_Visitor,False,False
12180,0,0.0,0,0.0,1,0.0,0.2,0.2,0.0,0.0,Dec,1,13,9,20,Returning_Visitor,False,False


In [9]:
df.info()


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 12330 entries, 0 to 12329
Data columns (total 18 columns):
 #   Column                   Non-Null Count  Dtype  
---  ------                   --------------  -----  
 0   Administrative           12330 non-null  int64  
 1   Administrative_Duration  12330 non-null  float64
 2   Informational            12330 non-null  int64  
 3   Informational_Duration   12330 non-null  float64
 4   ProductRelated           12330 non-null  int64  
 5   ProductRelated_Duration  12330 non-null  float64
 6   BounceRates              12330 non-null  float64
 7   ExitRates                12330 non-null  float64
 8   PageValues               12330 non-null  float64
 9   SpecialDay               12330 non-null  float64
 10  Month                    12330 non-null  object 
 11  OperatingSystems         12330 non-null  int64  
 12  Browser                  12330 non-null  int64  
 13  Region                   12330 non-null  int64  
 14  TrafficType           

In [10]:
df.describe()

,Administrative,Administrative_Duration,Informational,Informational_Duration,ProductRelated,ProductRelated_Duration,BounceRates,ExitRates,PageValues,SpecialDay,OperatingSystems,Browser,Region,TrafficType
count,12330.000000,12330.000000,12330.000000,12330.000000,12330.000000,12330.000000,12330.000000,12330.000000,12330.000000,12330.000000,12330.000000,12330.000000,12330.000000,12330.000000
mean,2.315166,80.818611,0.503569,34.472398,31.731468,1194.746220,0.022191,0.043073,5.889258,0.061427,2.124006,2.357097,3.147364,4.069586
std,3.321784,176.779107,1.270156,140.749294,44.475503,1913.669288,0.048488,0.048597,18.568437,0.198917,0.911325,1.717277,2.401591,4.025169
min,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,1.000000,1.000000,1.000000,1.000000
25%,0.000000,0.000000,0.000000,0.000000,7.000000,184.137500,0.000000,0.014286,0.000000,0.000000,2.000000,2.000000,1.000000,2.000000
50%,1.000000,7.500000,0.000000,0.000000,18.000000,598.936905,0.003112,0.025156,0.000000,0.000000,2.000000,2.000000,3.000000,2.000000
75%,4.000000,93.256250,0.000000,0.000000,38.000000,1464.157214,0.016813,0.050000,0.000000,0.000000,3.000000,2.000000,4.000000,4.000000
max,27.000000,3398.750000,24.000000,2549.375000,705.000000,63973.522230,0.200000,0.200000,361.763742,1.000000,8.000000,13.000000,9.000000,20.000000


In [11]:
df = df.drop_duplicates().reset_index(drop = True)

In [12]:
df.head(1)

,Administrative,Administrative_Duration,Informational,Informational_Duration,ProductRelated,ProductRelated_Duration,BounceRates,ExitRates,PageValues,SpecialDay,Month,OperatingSystems,Browser,Region,TrafficType,VisitorType,Weekend,Revenue
0,0,0.0,0,0.0,1,0.0,0.2,0.2,0.0,0.0,Feb,1,1,1,1,Returning_Visitor,False,False


In [22]:
seq_cols = ['Administrative' , 'Administrative_Duration', 'Informational' , 'Informational_Duration' , 'ProductRelated' ,'ProductRelated_Duration' ]

In [13]:
num_cols = ['BounceRates' , 'ExitRates' , 'PageValues' , 'SpecialDay']

In [14]:
cat_cols = ['Month' , 'OperatingSystems' , 'Browser' , 'Region' , 'TrafficType' , 'VisitorType' , 'Weekend']

In [15]:
target_col = ['Revenue']

In [16]:
X , y = df.drop(columns='Revenue') , df['Revenue'].astype(int).values.reshape(-1 , 1)

In [17]:
X_train , X_test , y_train , y_test = train_test_split(X , y , test_size=0.20 , random_state=42 , stratify=y)
X_tr , X_val , y_tr , y_val = train_test_split(X_train , y_train , test_size = 0.15 ,random_state=42, stratify=y_train)

In [18]:
device = 'cuda' if torch.cuda.is_available() else 'cpu'

In [19]:
transformer = ColumnTransformer(transformers=[
    ('num' , StandardScaler() , num_cols),
    ('cat' , OneHotEncoder(drop='first' , sparse_output=False , handle_unknown='ignore') , cat_cols)
    ])

In [20]:
X_ctx_train = transformer.fit_transform(X_train)
X_ctx_val = transformer.transform(X_val)
X_ctx_test = transformer.transform(X_test)

/usr/local/lib/python3.13/dist-packages/sklearn/preprocessing/_encoders.py:246: UserWarning: Found unknown categories in columns [4] during transform. These unknown categories will be encoded as all zeros
  warnings.warn(


In [23]:
scaler = StandardScaler()
X_seq_train = scaler.fit_transform(X_train[seq_cols]).reshape(-1 , 3 , 2)
X_seq_val = scaler.transform(X_val[seq_cols]).reshape(-1 , 3 , 2)
X_seq_test = scaler.transform(X_test[seq_cols]).reshape(-1 , 3 , 2)

In [24]:
print(X_ctx_train.shape)
print(X_seq_train.shape)
contx_dim = X_ctx_train.shape[1]

(9764, 61)
(9764, 3, 2)


In [25]:
class CustomDataset(Dataset):
  def __init__(self , seq , ctx , labels):
    self.seq = torch.tensor(seq , dtype = torch.float32)
    self.ctx = torch.tensor(ctx , dtype = torch.float32)
    self.labels = torch.tensor(labels , dtype = torch.float32)
  def __len__(self):
    return len(self.labels)
  def __getitem__(self , idx):
    return self.seq[idx] , self.ctx[idx] , self.labels[idx]

In [26]:
train_dataset = CustomDataset(X_seq_train , X_ctx_train , y_train)
val_dataset = CustomDataset(X_seq_val , X_ctx_val, y_val )
test_dataset = CustomDataset(X_seq_test , X_ctx_test , y_test)

In [27]:
train_loader = DataLoader(train_dataset , batch_size = 64 , shuffle = True)
val_loader = DataLoader(val_dataset , batch_size = 64 , shuffle = False)
test_loader = DataLoader(test_dataset , batch_size = 128 , shuffle = False)

In [31]:
pos_weight_val = (y_train == 0).sum() / (y_train == 1).sum()
pos_weight_train = torch.tensor([pos_weight_val], dtype=torch.float32, device=device)

In [32]:
class LstmModel(nn.Module):
  def __init__(self , seq_features , lstm_hidden , ctx_hidden , context_dim , dropout_rate):
    super().__init__()
    self.lstm = nn.LSTM(
        input_size = seq_features,
        hidden_size = lstm_hidden,
        num_layers = 1,
        batch_first = True
    )
    self.context_net = nn.Sequential(
        nn.Linear(context_dim , ctx_hidden),
        nn.BatchNorm1d(ctx_hidden),
        nn.ReLU(),
        nn.Dropout(dropout_rate)
    )
    self.classifier = nn.Sequential(
        nn.Linear(lstm_hidden + ctx_hidden , 32),
        nn.ReLU(),
        nn.Dropout(dropout_rate),
        nn.Linear(32 , 1)
    )

  def forward(self , X_seq , X_ctx):
    _, (h_n , c_n) = self.lstm(X_seq)
    seq_features = h_n[-1]

    ctx_features = self.context_net(X_ctx)

    fused = torch.cat([seq_features , ctx_features], dim = 1)

    logits = self.classifier(fused)

    return logits

In [33]:
def objective(trial):
  lstm_hidden = trial.suggest_categorical('lstm_hidden' , [32 , 64 , 128])
  ctx_hidden = trial.suggest_categorical('ctx_hidden' , [32 , 64])
  dropout_rate = trial.suggest_float('dropout_rate' , 0.2 , 0.4)
  lr = trial.suggest_float('lr' , 1e-4 , 5e-3 , log = True)
  weight_decay = trial.suggest_float('weight_decay' , 1e-5 , 1e-2 , log = True)
  batch_size = trial.suggest_categorical('batch_size' , [32 , 64 , 128])

  train_dataset = CustomDataset(X_seq_train , X_ctx_train , y_train)
  train_loader = DataLoader(train_dataset , batch_size=batch_size, shuffle=True)

  model = LstmModel(seq_features=2 ,
                    lstm_hidden=lstm_hidden ,
                    ctx_hidden = ctx_hidden ,
                    context_dim= X_ctx_train.shape[1],
                    dropout_rate=dropout_rate).to(device)
  criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weight_train)
  optimizer = torch.optim.Adam(model.parameters(), lr = lr , weight_decay = weight_decay)

  best_val_auc = 0.0
  for epoch in range(15):
      model.train()
      for b_seq, b_ctx, b_labels in train_loader:
          b_seq, b_ctx, b_labels = b_seq.to(device), b_ctx.to(device), b_labels.to(device)
          optimizer.zero_grad()
          output = model(b_seq, b_ctx)
          loss = criterion(output, b_labels)
          loss.backward()
          optimizer.step()

      model.eval()
      val_probs, val_targets = [], []
      with torch.no_grad():
          for b_seq, b_ctx, b_labels in val_loader:
              b_seq, b_ctx, b_labels = b_seq.to(device), b_ctx.to(device), b_labels.to(device)
              logits = model(b_seq, b_ctx)
              probs = torch.sigmoid(logits).cpu().numpy()
              val_probs.extend(probs)
              val_targets.extend(b_labels.cpu().numpy())

      val_auc = roc_auc_score(np.array(val_targets).flatten(), np.array(val_probs).flatten())
      best_val_auc = max(best_val_auc, val_auc)

      trial.report(val_auc, epoch)
      if trial.should_prune():
          raise optuna.exceptions.TrialPruned()

  return best_val_auc

In [34]:
study = optuna.create_study(direction="maximize", pruner=optuna.pruners.MedianPruner(n_startup_trials=5, n_warmup_steps=4))
study.optimize(objective, n_trials=20)

[I 2026-11-17 14:23:11,304] A new study created in memory with name: no-name-ca34471a-fcd7-4f14-88aa-8572904015b9
[I 2026-11-17 14:23:32,535] Trial 0 finished with value: 0.9408077896016167 and parameters: {'lstm_hidden': 32, 'ctx_hidden': 64, 'dropout_rate': 0.2041968585571079, 'lr': 0.0037674656570123688, 'weight_decay': 0.00021866755520651588, 'batch_size': 32}. Best is trial 0 with value: 0.9408077896016167.
[I 2026-11-17 14:23:49,741] Trial 1 finished with value: 0.9367236189426379 and parameters: {'lstm_hidden': 128, 'ctx_hidden': 64, 'dropout_rate': 0.2234650589433447, 'lr': 0.000622462042190277, 'weight_decay': 0.00038588325253038647, 'batch_size': 64}. Best is trial 0 with value: 0.9408077896016167.
[I 2026-11-17 14:24:06,700] Trial 2 finished with value: 0.9163840250985712 and parameters: {'lstm_hidden': 128, 'ctx_hidden': 32, 'dropout_rate': 0.31229942844722786, 'lr': 0.002510688547121834, 'weight_decay': 0.008299542429554078, 'batch_size': 64}. Best is trial 0 with value: 0

In [35]:
 best = study.best_params
 best

{'lstm_hidden': 64,
 'ctx_hidden': 64,
 'dropout_rate': 0.2318904397959229,
 'lr': 0.0034754541202673267,
 'weight_decay': 1.2573282772383974e-05,
 'batch_size': 64}

In [36]:
full_train_loader = DataLoader(
    CustomDataset(X_seq_train, X_ctx_train, y_train),
    batch_size=best["batch_size"],
    shuffle=True,
    drop_last=True
)
final_test_loader = DataLoader(
    CustomDataset(X_seq_test, X_ctx_test, y_test),
    batch_size=128,
    shuffle=False
)

In [37]:
pos_weight_full = torch.tensor([(y_train == 0).sum() / (y_train == 1).sum()], dtype=torch.float32, device=device)

In [38]:
final_model = LstmModel(
    seq_features=2,
    lstm_hidden=best["lstm_hidden"],
    ctx_hidden=best["ctx_hidden"],
    context_dim=X_ctx_train.shape[1],
    dropout_rate=best["dropout_rate"]
).to(device)

In [39]:
criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weight_full)
optimizer = torch.optim.AdamW(final_model.parameters(), lr=best["lr"], weight_decay=best["weight_decay"])
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode="min", factor=0.5, patience=3)

In [40]:
for epoch in range(1, 21):
    final_model.train()
    running_loss = 0.0

    for b_seq, b_ctx, b_labels in full_train_loader:
        b_seq, b_ctx, b_labels = b_seq.to(device), b_ctx.to(device), b_labels.to(device)

        optimizer.zero_grad()
        loss = criterion(final_model(b_seq, b_ctx), b_labels)
        loss.backward()
        optimizer.step()

        running_loss += loss.item()

    avg_loss = running_loss / len(full_train_loader)
    scheduler.step(avg_loss)
    print(f"Epoch [{epoch:02d}/20] | Loss: {avg_loss:.4f}")

Epoch [01/20] | Loss: 0.7728
Epoch [02/20] | Loss: 0.6620
Epoch [03/20] | Loss: 0.6484
Epoch [04/20] | Loss: 0.6250
Epoch [05/20] | Loss: 0.6085
Epoch [06/20] | Loss: 0.6061
Epoch [07/20] | Loss: 0.6044
Epoch [08/20] | Loss: 0.5921
Epoch [09/20] | Loss: 0.5761
Epoch [10/20] | Loss: 0.5557
Epoch [11/20] | Loss: 0.5718
Epoch [12/20] | Loss: 0.5705
Epoch [13/20] | Loss: 0.5489
Epoch [14/20] | Loss: 0.5456
Epoch [15/20] | Loss: 0.5412
Epoch [16/20] | Loss: 0.5374
Epoch [17/20] | Loss: 0.5405
Epoch [18/20] | Loss: 0.5327
Epoch [19/20] | Loss: 0.5210
Epoch [20/20] | Loss: 0.5149


In [41]:
final_model.eval()
test_probs, test_actuals = [], []

with torch.no_grad():
    for b_seq, b_ctx, b_labels in final_test_loader:
        probs = torch.sigmoid(final_model(b_seq.to(device), b_ctx.to(device))).cpu().numpy()
        test_probs.extend(probs)
        test_actuals.extend(b_labels.numpy())

y_probs = np.array(test_probs).flatten()
y_true = np.array(test_actuals).flatten()
y_pred = (y_probs >= 0.5).astype(int)
print(f"FINAL TEST ROC-AUC SCORE: {roc_auc_score(y_true, y_probs):.4f}")
print("\nConfusion Matrix:\n", confusion_matrix(y_true, y_pred))
print("\nClassification Report:\n", classification_report(y_true, y_pred, target_names=["Drop-off (0)", "Purchase (1)"], digits=4))

FINAL TEST ROC-AUC SCORE: 0.9226

Confusion Matrix:
 [[1743  316]
 [  59  323]]

Classification Report:
               precision    recall  f1-score   support

Drop-off (0)     0.9673    0.8465    0.9029      2059
Purchase (1)     0.5055    0.8455    0.6327       382

    accuracy                         0.8464      2441
   macro avg     0.7364    0.8460    0.7678      2441
weighted avg     0.8950    0.8464    0.8606      2441



In [42]:
os.makedirs('artifacts' , exist_ok=True)
checkpoint = {
    'model_state_dict' : final_model.state_dict(),
    'hyperparameteres': {
        'seq_features' :2,
        'lstm_hidden' : best['lstm_hidden'],
        'ctx_hidden': best['ctx_hidden'],
        'context_dim':X_ctx_train.shape[1],
        'dropout_rate' : best['dropout_rate']
    }
}
torch.save(checkpoint, "artifacts/session_intent_lstm.pt")
joblib.dump(scaler, "artifacts/seq_scaler.joblib")
joblib.dump(transformer, "artifacts/context_transformer.joblib")

print("Saved all deployment artifacts to ./artifacts/")

Saved all deployment artifacts to ./artifacts/
